# Solving Pickle Doomsday

In [1]:
from oggm import cfg, utils
from oggm import workflow
from oggm import DEFAULT_BASE_URL
from oggm import GlacierDirectory

import os
import geopandas as gpd
import xarray as xr
import xvec
from oggmzarr.datacube.geozarr import OggmZarrHandler, GeoZarrHandler

In [2]:
cfg.initialize(logging_level="ERROR")

cfg.PARAMS["use_multiprocessing"] = True
cfg.PARAMS["continue_on_error"] = True
cfg.PATHS['working_dir'] = utils.gettempdir(dirname='oggm-geozarr', reset=True)
rgi_ids = ['RGI60-11.00897', "RGI60-06.00377"] # HEF, Bruarjoekull 

DEFAULT_BASE_URL

2026-04-22 14:23:26: oggm.cfg: Reading default parameters from the OGGM `params.cfg` configuration file.
2026-04-22 14:23:26: oggm.cfg: Multiprocessing switched OFF according to the parameter file.
2026-04-22 14:23:26: oggm.cfg: Multiprocessing: using all available processors (N=22)
2026-04-22 14:23:26: oggm.cfg: Multiprocessing switched ON after user settings.
2026-04-22 14:23:26: oggm.cfg: PARAMS['continue_on_error'] changed from `False` to `True`.


'https://cluster.klima.uni-bremen.de/~oggm/gdirs/oggm_v1.6/L3-L5_files/2025.6/elev_bands/W5E5/per_glacier_spinup/'

In [3]:
gdirs = workflow.init_glacier_directories(
    rgi_ids,
    prepro_base_url=DEFAULT_BASE_URL,  # where to fetch the data?
    from_prepro_level=4,  # what kind of data? 
    prepro_border=80  # how big of a map?
)
gdir = gdirs[0]

2026-04-22 14:23:46: oggm.workflow: init_glacier_directories from prepro level 4 on 2 glaciers.
2026-04-22 14:23:46: oggm.workflow: Execute entity tasks [gdir_from_prepro] on 2 glaciers


In [15]:
from pathlib import Path
pickle_files = [Path(f) for f in os.listdir(gdir.dir) if f[-4:]==".pkl"]
pickle_files


[PosixPath('inversion_flowlines.pkl'),
 PosixPath('inversion_input.pkl'),
 PosixPath('downstream_line.pkl'),
 PosixPath('model_flowlines.pkl'),
 PosixPath('inversion_output.pkl'),
 PosixPath('model_flowlines_dyn_melt_f_calib.pkl')]

In [44]:
def get_tranche(data:dict):
    tranche = {}
    for k, v in i.items():
        tranche[k] = type(v)
    return tranche

pickle_data = {}
for pickle in pickle_files:
    try:
        stem = gdir.read_pickle(pickle.stem)
        if isinstance(stem, list):
            slices = []
            for i in stem:
                if isinstance(i, dict):
                    slices.append(get_tranche(i))
                else:
                    slices.append(type(i))
            pickle_data[pickle.stem] = slices
        elif isinstance(stem, dict):        
            pickle_data[pickle.stem] = get_tranche(i)
    except:
        print(f"Pickle {pickle.stem} not found.")

Pickle model_flowlines_dyn_melt_f_calib not found.


In [86]:
def text_to_dict(lines):
    v_text = ["| variable| type | description |\n|---|---|---|"]
    v_text.append(f"{"\n  - ".join([f"<b>{i}</b>: {str(j)[7:-1]}" for i,j in lines.items()])}")
    return v_text

In [88]:
import pprint
# pprint.pp(pickle_data, sort_dicts=True)
text = []
# text.append(f"| pickle file | type | description |")
for k,v in pickle_data.items():
    if isinstance(v, list):
        if isinstance(v[0], dict):
            v_text = text_to_dict(v[0])
        else:
            v_text = f"{v}"
    elif isinstance(v, dict):
        v_text = text_to_dict(v)
    else:
        v_text = f"{v}"
    text.append(f"| <b>{k}</b> | {v_text}| |\n")
print("\n".join(text))

| <b>inversion_flowlines</b> | [<class 'oggm.core.centerlines.Centerline'>]| |

| <b>inversion_input</b> | ['| variable| type | description |\n|---|---|---|', "<b>dx</b>: 'float'\n  - <b>flux_a0</b>: 'numpy.ndarray'\n  - <b>width</b>: 'numpy.ndarray'\n  - <b>slope_angle</b>: 'numpy.ndarray'\n  - <b>is_rectangular</b>: 'numpy.ndarray'\n  - <b>is_trapezoid</b>: 'numpy.ndarray'\n  - <b>flux</b>: 'numpy.ndarray'\n  - <b>is_last</b>: 'bool'\n  - <b>hgt</b>: 'numpy.ndarray'\n  - <b>invert_with_trapezoid</b>: 'bool'"]| |

| <b>downstream_line</b> | ['| variable| type | description |\n|---|---|---|', "<b>dx</b>: 'float'\n  - <b>flux_a0</b>: 'numpy.ndarray'\n  - <b>width</b>: 'numpy.ndarray'\n  - <b>slope_angle</b>: 'numpy.ndarray'\n  - <b>is_rectangular</b>: 'numpy.ndarray'\n  - <b>is_trapezoid</b>: 'numpy.ndarray'\n  - <b>flux</b>: 'numpy.ndarray'\n  - <b>is_last</b>: 'bool'\n  - <b>hgt</b>: 'numpy.ndarray'\n  - <b>invert_with_trapezoid</b>: 'bool'"]| |

| <b>model_flowlines</b> | [<class 'og